In [1]:
import numpy as np
import pandas as pd
import torch

from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[0] / '2_Propensities'))

import MF_class as MF

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

```
                                USERS                             
         ┌───────────────────────────────────────────────────────┐
         │                          │                            │
         │                          │                            │
         │                          │                            │
         │                          │                            │
ITEMS    │                          │                            │
         │                          │                            │
         │                          │                            │
         ├──────────────────────────┼────────────────────────────┤
         │                          │████████████████████████████│
         │                          │████████████████████████████│
         └───────────────────────────────────────────────────────┘
```

# 1. Load Data

In [2]:
base_artifacts = Path.cwd().resolve().parents[1] / 'CausalI2I_artifacts'

train = pd.read_csv(base_artifacts / 'Datasets' / 'Simulation' / 'train.csv')
test = pd.read_csv(base_artifacts / 'Datasets' / 'Simulation' / 'test.csv')

n_users = train['user_id'].nunique()
n_items = train['item_id'].nunique()
print(f'Number of users: {n_users}, Number of items: {n_items}')

Number of users: 6040, Number of items: 3952


# 2. Train Model

In [3]:
model = MF.MatrixFactorizationTorch(n_users, n_items, n_factors=20)
model.fit(
    train_data=train.values,
    val_data=test.values,
    lr=2e-3, 
    wd=1e-7,
    pos_weight=1,
    batch_size=2**15,
    n_epochs=50,
    device=torch.device('cuda:0'), 
    use_amp=True)

Epoch  ||- - - - - - - - Train - - - - - - - -||- - - - - - Validation - - - - - - - || Epoch's | COS θ | Time     
Number || BCE    | BCE-POS | BCE-NEG | MPR    || BCE    | BCE-POS | BCE-NEG | MPR    || Change  |       | Elapsed  
=======||========|=========|=========|========||========|=========|=========|========||=========|=======|==========
   1   || 0.2346 |  1.8846 |  0.0787 | 0.7961 || 0.2425 |  1.8492 |  0.0842 | 0.7927 || 155.61  | None  | 00:04.78
   2   || 0.2249 |  1.7633 |  0.0795 | 0.8077 || 0.2325 |  1.7516 |  0.0828 | 0.8035 ||  48.50  | 0.218 | 00:09.12
   3   || 0.2209 |  1.7274 |  0.0785 | 0.8138 || 0.2292 |  1.7216 |  0.0822 | 0.8079 ||  28.99  | 0.553 | 00:13.57
   4   || 0.2176 |  1.6989 |  0.0776 | 0.8186 || 0.2266 |  1.7004 |  0.0814 | 0.8114 ||  25.09  | 0.721 | 00:18.07
   5   || 0.2152 |  1.6795 |  0.0768 | 0.8221 || 0.2248 |  1.6882 |  0.0806 | 0.8139 ||  21.06  | 0.770 | 00:22.52
   6   || 0.2132 |  1.6605 |  0.0764 | 0.8250 || 0.2235 |  1.6713 |  0.0809 |

### Save Model

In [4]:
model.save(path=base_artifacts / 'Propensity_Models' / 'MF_simulation.pt', note=None)

### Load Model

In [5]:
loaded_model = MF.MatrixFactorizationTorch(n_users, n_items, n_factors=20)
loaded_model.load(path=base_artifacts / 'Propensity_Models' / 'MF_simulation.pt')

Loaded model summary:
Model:                      MatrixFactorizationTorch
Number of users:            6040
Number of items:            3952
Number of factors:          20
Learning rate:              0.002
Weight decay:               1e-07
Positive weight:            1
Batch size:                 32768
Number of epochs:           50
Device:                     cuda:0
Use AMP:                    True
Timestamp:                  2026-04-11 14:29:10
